# Banking loan default — random forest

Use the same **1,000-row synthetic CSV** and the same train/test split as the decision tree lessons. Put `Banking_Loan_Default_Classification_1000_Records.csv` beside this notebook. Run in order. The generated outcomes follow a simulated risk rule with random variation; measured scores apply to this teaching data.

## What is a random forest?

A decision tree asks a sequence of questions. A random forest trains many trees on different samples of training applicants and random subsets of input columns. The trees vote for a class; their combined votes produce an estimated default probability. This often reduces the overfitting of a single deep tree.

## 1. Read data and do basic EDA

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, roc_auc_score, confusion_matrix,
                             ConfusionMatrixDisplay, RocCurveDisplay, classification_report)

df = pd.read_csv("Banking_Loan_Default_Classification.csv")
display(df.head())
print("Shape:", df.shape)
print("Missing values:", df.isna().sum().sum())
display(df.describe(include="all").T)

## 2. Check class balance

Count the default (1) and no-default (0) records. A majority-only guess scores about 61% on this generated dataset; use it as a reference, not a useful loan-risk model.

In [ ]:
print(df["loan_default"].value_counts())
print((df["loan_default"].value_counts(normalize=True)*100).round(1))
df["loan_default"].value_counts().sort_index().plot.bar(rot=0, title="Loan default labels")
plt.ylabel("Applicants")
plt.show()

## 3. One-hot encode and keep the same 80/20 split

Each text category becomes a 0/1 indicator. `loan_default` stays separate as the target. The 200 test records are held out from training and GridSearchCV.

In [ ]:
X = pd.get_dummies(df.drop(columns="loan_default"), dtype=int)
y = df["loan_default"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Encoded features:", X.shape[1])
print("Training rows:", len(X_train), "Test rows:", len(X_test))
display(X_train.head())

## 4. Rebuild the first two models on the shared training split

In [ ]:
basic_model = RandomForestClassifier(random_state=42)
basic_model.fit(X_train, y_train)
manual_model = RandomForestClassifier(n_estimators=200, max_depth=9,
                                      random_state=42, n_jobs=-1)
manual_model.fit(X_train, y_train)

## 5. GridSearchCV: select settings using training folds

Try a small grid of tree counts, depths, minimum leaf sizes, and feature sampling values. Four-fold stratified cross-validation repeats training/validation within the 800 training rows. The 200 test rows do not select parameters. `best_score_` is cross-validation accuracy, not final test accuracy.

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

parameter_grid = {
    "n_estimators": [100, 200],
    "max_depth": [5, 8, None],
    "min_samples_leaf": [1, 3, 5],
    "max_features": ["sqrt", 0.7]
}
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)
search = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    parameter_grid, scoring="accuracy", cv=cv, n_jobs=-1, refit=True
)
search.fit(X_train, y_train)
print("Best training CV accuracy:", round(search.best_score_, 3))
print("Selected parameters:", search.best_params_)
model = search.best_estimator_

### Top candidates during training cross-validation

In [ ]:
cv_table = pd.DataFrame(search.cv_results_)
display(cv_table[["mean_test_score", "std_test_score", "params"]]
        .sort_values("mean_test_score", ascending=False).head(10).round(3))

## Evaluate the current forest

The confusion matrix has actual outcomes in rows and predicted outcomes in columns. A false negative is a missed default; a false positive is a false alarm. Accuracy is the fraction of correct classes. ROC AUC evaluates how default probabilities rank the two classes across thresholds.

In [ ]:
pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:, 1]
print("Training accuracy:", round(model.score(X_train, y_train), 3))
print("Test accuracy:", round(accuracy_score(y_test, pred), 3))
print("Test ROC AUC:", round(roc_auc_score(y_test, prob), 3))
print(classification_report(y_test, pred, target_names=["No default", "Default"], zero_division=0))
display(pd.DataFrame(confusion_matrix(y_test, pred, labels=[0,1]),
    index=["Actual no default", "Actual default"],
    columns=["Predicted no default", "Predicted default"]))
ConfusionMatrixDisplay.from_predictions(y_test, pred, labels=[0,1], display_labels=["No default", "Default"])
plt.show()
RocCurveDisplay.from_predictions(y_test, prob)
plt.show()

### Read the result

Compare training and test accuracy for possible overfitting. The test set has 200 applicants, so one changed prediction shifts accuracy by 0.5 percentage points. ROC AUC uses probabilities, which need not move in exactly the same way as accuracy.

## Which inputs did the forest use?

This chart shows impurity-based importance. Correlated inputs can share importance; these bars do not establish causes of default.

In [ ]:
importance = pd.Series(model.feature_importances_, index=X.columns)
importance.nlargest(12).sort_values().plot.barh(figsize=(8, 5))
plt.title("Top forest input importances")
plt.xlabel("Relative importance")
plt.show()

## Compare all three models

Report measured held-out scores. Grid search can choose a model with a lower test score on other datasets: cross-validation only estimates which training candidate may generalize best. This dataset is synthetic and selected for teaching.

In [ ]:
rows = []
for name, fitted in [("Basic", basic_model),
                     ("Manual hyperparameters", manual_model),
                     ("GridSearchCV selected", model)]:
    rows.append({"Model": name,
                 "Train accuracy": fitted.score(X_train, y_train),
                 "Test accuracy": fitted.score(X_test, y_test),
                 "Test ROC AUC": roc_auc_score(y_test, fitted.predict_proba(X_test)[:, 1])})
display(pd.DataFrame(rows).round(3))